In [ ]:
# Minimal valid match statement (should compile)
from guppylang import guppy
from typing import Generic

T = guppy.type_var("T")

@guppy.enum
class EnumGen(Generic[T]):      # pyright: ignore[reportInvalidTypeForm]
    Var = {"A": T}

@guppy.enum
class Enum:
    North = {"A": int}
    South = {"B": int}
    East = {}
    West = {}

@guppy.struct
class Point(Generic[T]):         # pyright: ignore[reportInvalidTypeForm] 
    x: T                         # pyright: ignore[reportInvalidTypeForm] 
    y: int

@guppy
def main(north: Enum, point: Point[int]) -> int:
    match north:
        case Enum.North(1):
            x = 1
        case Enum.South(_):
            x = 2
        case _:
            x = 0
    
    a = Enum.North

    match a(7):
        case Enum.North(_):
            y = 3
        case _:
            y = 0
    

    match Point(3, 5):
        case Point(7, _):
            z = 4
        case _:
            z = 0
    
    match point:
        case Point(7, _):
            w = 5
        case _:
            w = 0

    match Point(EnumGen.Var(3), 5):
        case Point(EnumGen.Var(7), _):
            pass

    # Match on integer
    match 42:
        case 1:
            int_result = "one"
        case 42:
            int_result = "forty-two"
        case _:
            int_result = "other"

    # Match on string
    match "guppy":
        case "guppy":
            str_result = "fish"
        case "shark":
            str_result = "predator"
        case _:
            str_result = "unknown"
    


    return 0

main.check()

In [30]:
from guppylang.decorator import guppy
from guppylang.std.builtins import owned
from guppylang.std.quantum import qubit, measure, h

@guppy.struct
class Point:
    x: int
    y: int


@guppy.struct
class MyStruct:
    q1: qubit
    i: int


@guppy
def foo(s1: MyStruct @owned, s2: MyStruct, p: Point) -> qubit:
    q = qubit()
    match s1:
        case MyStruct(_, 1):
            b = 2
        case _:
            b = 0
    h(s1.q1)
    h(s2.q1)
    match p:
        case Point(3, 4):
            measure(q)
            return s1.q1
        case Point(4, _):
            measure(q)
            return s1.q1
        case Point(5, _):
            measure(q)
            return s1.q1
    measure(q)
    match s2:
        case _:
            return s1.q1

foo.check()


In [23]:
# Example: linear enum
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, owned, h

@guppy.enum
class LinearEnum:
    var = {"a": qubit}


@guppy
def main(e: LinearEnum) -> None:
    match e:
        case LinearEnum.var(_):
            pass

main.check()

### Linearity Error Tests - Expected to Fail

In [10]:
from guppylang import guppy
from guppylang.std.quantum import qubit

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def main(p: Point) -> None:
    match p:
        case Point(_, _):
            b = p.x
        case Point(_, 1):
            pass
        case _:
            c = p.x


main.check()

Error: Copy violation (at <In[10]>:10:9)
   | 
 8 | 
 9 | @guppy
10 | def main(p: Point) -> None:
   |          ^^^^^^^^ Borrowed argument p cannot be returned to the caller ...

Note:
   | 
12 |         case Point(_, _):
13 |             b = p.x
   |                 --- since `p.x` with non-copyable type `qubit` was already moved
   |                     here

Help: Consider writing a value back into `p.x` before returning

Guppy compilation failed due to 1 previous error


In [11]:
@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def describe_point(point: Point)-> Point:
    return Point(qubit(), 3)

@guppy
def main() -> None:
    match describe_point(fun()):
        case _:
            pass



main.check()

Error: Drop violation (at <In[11]>:16:25)
   | 
14 | @guppy
15 | def main() -> None:
16 |     match describe_point(fun()):
   |                          ^^^^^ Value with non-droppable type `Point` would be leaked after
   |                                `describe_point` returns

Help: Consider assigning the value to a local variable before passing it to
`describe_point`

Guppy compilation failed due to 1 previous error


In [ ]:
# TODO: NICOLa Fix the errror rendering
from guppylang import guppy
from guppylang.std.quantum import qubit

@guppy.struct
class Point:
    x: qubit
    y: int


@guppy
def main(p: Point @owned) -> None:
    match p:
        case Point(_, _):
            b = p.x


main.check()

Error: Drop violation (at <In[2]>:15:12)
   | 
13 |     match p:
14 |         case _:
15 |             b = p.x
   |             ^ Variable `b` with non-droppable type `qubit` is leaked

Help: Make sure that `b` is consumed or returned to avoid the leak

Guppy compilation failed due to 1 previous error


In [3]:
# Example: match on linear variable
from guppylang import guppy
from guppylang.std.quantum import qubit, owned

@guppy.enum
class LinearEnum:
    var = {"a": qubit}


@guppy
def main(e: LinearEnum @owned) -> None:
    match e:
        case LinearEnum.var(_):
            pass

main.check()

Error: Drop violation (at <In[3]>:11:9)
   | 
 9 | 
10 | @guppy
11 | def main(e: LinearEnum @owned) -> None:
   |          ^^^^^^^^^^^^^^^^^^^^ Variable `e` with non-droppable type `LinearEnum` is leaked

Help: Make sure that `e` is consumed or returned to avoid the leak

Guppy compilation failed due to 1 previous error


In [4]:
# ERROR Test 4: Linear value created in arm not consumed
from guppylang import guppy
from guppylang.std.quantum import qubit

@guppy.struct
class Point:
    x: int
    y: int

@guppy
def test_linear_not_consumed_in_arm(p: Point) -> int:
    match p:
        case Point(3, _):
            q = qubit()  # ERROR: q created but not consumed before arm ends
            result = 1
        case Point(_, 4):
            result = 2
        case _:
            result = 0
    
    return result

# This should error - q is not consumed/measured before the arm ends
test_linear_not_consumed_in_arm.check()

Error: Drop violation (at <In[4]>:14:12)
   | 
12 |     match p:
13 |         case Point(3, _):
14 |             q = qubit()  # ERROR: q created but not consumed before arm ends
   |             ^ Variable `q` with non-droppable type `qubit` is leaked

Help: Make sure that `q` is consumed or returned to avoid the leak

Guppy compilation failed due to 1 previous error
